# RSim 原型：直接使用 Notebook 的 asyncio 循环
先在启动 Jupyter 的终端 `conda activate rsim`，选择 **rsim ROS2** kernel。
本 Notebook 包含真实硬件验证：当前 UVC 相机和 192.168.199.97 的 Robin W。
每个资源图用 `async with Runtime(...)` 管理，单元格结束即释放本客户端资源。

In [1]:
%load_ext autoreload
%autoreload 3 --print --log

In [ ]:
import asyncio, os, sys
import numpy as np
from rsim import Runtime, ProcessSensor, Map, Bundle, RobinW, allocate
from rsim.drivers import Camera, RobinW as RobinWDriver
from rsim.synthetic import CounterArray

from scipykit.task.cv import *
os.environ['NOTEBOOK_ASSETS_ROOT'] = '../assets'
os.chdir(Path(os.getcwd()).parent)

assert sys.version_info[:2] == (3, 12)
print(sys.executable)
print(type(asyncio.get_running_loop()).__name__)

/home/root/LAB/envs/rsim/bin/python
_UnixSelectorEventLoop


## Notebook 函数 + 两层子进程 + 共享输出
这里的函数和 lambda 在 Notebook 内定义；传感器图在子进程中创建。
`allocate` 让 NumPy 直接写入可共享的输出，返回主进程时映射其数组。

In [6]:
def squared(values):
    output = allocate(values.shape, values.dtype)
    np.square(values, out=output)
    return output

sensor = ProcessSensor(lambda: Map(
    ProcessSensor(lambda: CounterArray(size=2048)), squared, hz=20))
async with Runtime(sensor):
    frame = await sensor.get(timeout=20)
    same = await sensor.get(timestamp_ns=frame.stamp_ns, clock=frame.clock)
    assert same is frame
    assert isinstance(frame.data, np.memmap) and not frame.data.flags.writeable
    retained_value = float(frame.data[0])
    print({'main_pid': os.getpid(), 'worker_pid': sensor.worker_pid,
           'shape': frame.data.shape, 'value': retained_value})
assert float(frame.data[0]) == retained_value
assert not sensor.directory.exists()

{'main_pid': 439518, 'worker_pid': 439687, 'shape': (2048,), 'value': 1.0}


## 真实硬件与共享雷达
相机当前为 Orbbec C8011 的 UVC 彩色图；D435 未连接。
`processing` 内再次使用 `RobinW()`，它与主进程的 lidar 复用同一个采集进程。
Bundle 保留各设备时间域，此处不做跨设备时间同步或空间标定。

In [11]:
def processing():
    source = RobinW()
    def compute(cloud):
        points = cloud.points
        xyz = np.stack([points[k].ravel() for k in ('x', 'y', 'z')], axis=1)
        finite = xyz[np.isfinite(xyz).all(axis=1)]
        voxels, counts = np.unique(np.floor(finite / 0.1).astype(np.int32),
                                   axis=0, return_counts=True)
        return {'voxels': voxels, 'counts': counts, 'source_pid': source.worker_pid}
    return Map(source, compute, hz=10)
lidar = RobinWDriver()
rig = Bundle(camera=Camera(), lidar=lidar, voxels=ProcessSensor(processing), hz=10)
async with Runtime(rig):
    samples = (await rig.get(timeout=30)).data
    image = samples['camera']['data'].pixels
    cloud = samples['lidar']['data'].points
    stats = samples['voxels']['data']
    assert stats['source_pid'] == lidar.worker_pid
    print({'camera': image.shape, 'points': cloud.size,
           'voxels': len(stats['voxels']), 'shared_lidar_pid': lidar.worker_pid})

show_image(image, key='tmp')
# ok, jpeg = cv2.imencode('.jpg', image)
# assert ok

# disp(data=jpeg.tobytes(), width=320)

{'camera': (640, 640, 3), 'points': 91474, 'voxels': 4316, 'shared_lidar_pid': 441613}


../prototype.ipynb/tmp.png